In [1]:
import pandas as pd
import folium 
import branca 
from IPython.display import HTML 
import plotly.express as px
from IPython.display import display

In [2]:
data =  pd.read_csv('C:\\Users\\HP\\Desktop\\Master2\\ML\\m2_enedis_dpe_app-main\\data\\df_adem_enedis_iris_69_prepared.csv')

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358302 entries, 0 to 358301
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   emission_ges_5_usages              358302 non-null  float64
 1   nom_commune_ban                    358302 non-null  object 
 2   type_energie_principale_chauffage  358302 non-null  object 
 3   qualite_isolation_murs             358302 non-null  object 
 4   type_batiment                      358302 non-null  object 
 5   conso_ecs_ep                       358302 non-null  float64
 6   surface_habitable_logement         358302 non-null  float64
 7   conso_chauffage_ep                 358302 non-null  float64
 8   isolation_toiture                  358302 non-null  float64
 9   etiquette_dpe                      358302 non-null  object 
 10  code_postal_ban                    358302 non-null  int64  
 11  zone_climatique                    3583

In [4]:
data.isnull().sum()

emission_ges_5_usages                0
nom_commune_ban                      0
type_energie_principale_chauffage    0
qualite_isolation_murs               0
type_batiment                        0
conso_ecs_ep                         0
surface_habitable_logement           0
conso_chauffage_ep                   0
isolation_toiture                    0
etiquette_dpe                        0
code_postal_ban                      0
zone_climatique                      0
qualite_isolation_menuiseries        0
emission_ges_chauffage               0
conso_totale_mwh                     0
conso_moy_commune_mwh                0
lon                                  0
lat                                  0
conso_m2                             0
cout_m2                              0
anciennete                           0
volume_logement                      0
classe_annee_construction            0
color_dpe                            0
dtype: int64

In [5]:
data.columns

Index(['emission_ges_5_usages', 'nom_commune_ban',
       'type_energie_principale_chauffage', 'qualite_isolation_murs',
       'type_batiment', 'conso_ecs_ep', 'surface_habitable_logement',
       'conso_chauffage_ep', 'isolation_toiture', 'etiquette_dpe',
       'code_postal_ban', 'zone_climatique', 'qualite_isolation_menuiseries',
       'emission_ges_chauffage', 'conso_totale_mwh', 'conso_moy_commune_mwh',
       'lon', 'lat', 'conso_m2', 'cout_m2', 'anciennete', 'volume_logement',
       'classe_annee_construction', 'color_dpe'],
      dtype='object')

In [6]:
#travailler avec 5000 premiers échantillons
data = data.head(5000)

In [ ]:
# === 1. Import des librairies ===
import pandas as pd
import folium
from folium import FeatureGroup
from folium.plugins import MarkerCluster, FeatureGroupSubGroup, HeatMap
from IPython.display import display


# === 3. Carte centrée sur la moyenne des coordonnées ===
centre_lat, centre_lon = data["lat"].mean(), data["lon"].mean()
m = folium.Map(location=[centre_lat, centre_lon], zoom_start=9, tiles="CartoDB positron")

# === 4. Groupe principal + clustering ===
marker_cluster = MarkerCluster(name="Tous les logements").add_to(m) 

# === 5. Création de sous-groupes (par étiquette DPE) ===
for dpe in sorted(data["etiquette_dpe"].dropna().unique()):
    subgroup = FeatureGroupSubGroup(marker_cluster, f"DPE {dpe}")
    m.add_child(subgroup)

    subdata = data[data["etiquette_dpe"] == dpe]

    for _, row in subdata.iterrows():
        popup_html = f"""
        <b>Commune :</b> {row['nom_commune_ban']}<br>
        <b>Code postal :</b> {row['code_postal_ban']}<br>
        <b>DPE :</b> {row['etiquette_dpe']}<br>
        <b>Conso Totale :</b> {row['conso_totale_mwh']} MWh<br>
        <b>Conso/m² :</b> {row['conso_m2']}<br>
        <b>Type énergie :</b> {row['type_energie_principale_chauffage']}<br>
        <b>Qualité isolation murs :</b> {row['qualite_isolation_murs']}<br>
        <b>Zone climatique :</b> {row['zone_climatique']}<br>
        <b>Type bâtiment :</b> {row['type_batiment']}<br>
        """
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=5,
            color=row["color_dpe"],
            fill=True,
            fill_color=row["color_dpe"],
            fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(subgroup)

# === 6. Heatmap énergétique ===
heat_data = data[["lat", "lon", "conso_totale_mwh"]].dropna().values.tolist()
HeatMap(heat_data, name="Carte chaleur consommation", radius=15, blur=10).add_to(m)

# === 7. Ajout du contrôle des couches ===
folium.LayerControl(collapsed=False).add_to(m)

#enregistrement de la carte dans un fichier HTML
m.save("cartographie_energetique.html")  
